# Henrich's model: can a population maintain a difficult skill?

In earlier sessions, individuals copied traits at random or with a bias. Now, we are going to replicate [Henrich (2004)](https://www.cambridge.org/core/journals/american-antiquity/article/abs/demography-and-cultural-evolution-how-adaptive-cultural-processes-can-produce-maladaptive-lossesthe-tasmanian-case/8CD08CC61E6FACF59EC02659B2BDA43C) model of learning. Here everyone tries to copy the most skilled agent, but learning is imperfect. We will see why population size can determine whether skill improves or declines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# random number generator for reproducibility
rng = np.random.default_rng(101101)

## 1. One generation of copying

Suppose the best model has skill $z_m$. Each of $N$ learners tries to copy this person. A learner's new skill is

$$z_i = z_m + \epsilon_i, \qquad \epsilon_i \sim \mathrm{Gumbel}(-\alpha,\,\beta).$$

The most typical copying outcome is $z_m-\alpha$: larger $\alpha$ means a harder skill to copy. $\beta$ controls variation among learners, so a few may outperform the model. Each new population consists of $N$ learners. 

In [ ]:
N = 80 # number of learners
alpha = 5 # difficulty of learning
beta = 1 # variability in learning
z_model = 10 # initial skill of the model

copies = z_model + rng.gumbel(loc=-alpha, scale=beta, size=N)
print(f"Model skill: {z_model}")
print(f"Best learner: {copies.max():.2f}")

plt.figure(figsize=(6, 2.5))
plt.hist(copies, bins=18, color="lightgray", edgecolor="black")
plt.axvline(z_model, color="black", linestyle="--", label="Model")
plt.axvline(copies.max(), color="tab:blue", label="Best learner")
plt.xlabel("Skill")
plt.ylabel("Number of learners")
plt.legend(frameon=False, 
           loc="upper right",)
plt.xlim(None, 15)
sns.despine()
plt.show()

Most learners do worse than the model. How can the best learner sometimes do better? Try changing parameters in the cell above and inspect what happens with the learner distribution.

## 2. Repeat across generations

Start with everyone at skill 0. In each generation, all learners copy the current best individual (i.e. max skill of the learners distribution). Record the best skill, because this person becomes the model for the next generation.

The copying step uses the same equation as in Section 1.

In [ ]:
def simulate_henrich(N, alpha, beta, generations, rng):
    population = np.zeros(N)
    best_skill = [population.max()]

    for _ in range(generations):
        model = population.max()
        population = ...  # TODO: draw N copying outcomes around model
        best_skill.append(population.max())

    return np.array(best_skill)

In [ ]:
# A run has one starting value and one value per generation.
check = simulate_henrich(N=10, alpha=5, beta=1, 
                         generations=4, rng=rng)
assert check.shape == (5,) and check[0] == 0

One run is only one possible outcome. Compare many runs for three population sizes. With $\alpha=5$ and $\beta=1$, what happens below, near, and above $N=80$? 

In [ ]:
alpha = 5
beta = 1
generations = 50

plt.figure(figsize=(7, 4))
for N, color in [(20, "tab:blue"), (80, "tab:orange"), (300, "tab:green")]:
    for run in range(20):
        best = simulate_henrich(N, alpha, beta, generations, rng)
        plt.plot(best, color=color, alpha=0.25,
                 label=f"N = {N}" if run == 0 else None)

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Generation")
plt.ylabel("Best skill")
plt.legend(frameon=False, 
           loc="upper left")
sns.despine()
plt.show()

Does a large population guarantee improvement in every run? Why does repeating the simulation help us see the pattern? What happens when you change $\alpha$ or $\beta$?

## 3. Estimate the change by simulation

We can estimate the effect of population size without a formula. For each $N$, start everyone at skill 0, simulate **one generation**, and repeat this many times. The mean of the resulting best skills estimates the average change from the starting value of 0.

Compare the estimated change with zero: negative values mean skill tends to decline, and positive values mean it tends to improve. Individual runs can still go the other way.

We collect the best skill after one generation over `num_runs` independent runs for each population size.

In [ ]:
N_values = [10, 20, 40, 60, 80, 100, 150, 200, 300]
num_runs = 10_000
comparison_rng = np.random.default_rng(4)
mean_changes = []

for N in N_values:
    changes = ...  # TODO: repeat the simulation and collect the final best skill
    mean_changes.append(np.mean(changes))

plt.figure(figsize=(7, 4))
plt.plot(N_values, mean_changes, marker="o", color="black")
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Population size N")
plt.ylabel("Mean change in best skill after one generation")
sns.despine()
plt.show()

With these parameters, the estimated change crosses zero between about $N=80$ and $N=100$. This is an **average over many runs**, not a guarantee for any single population. Try increasing `num_runs`: does the curve become more stable?

How would increasing $\alpha$ shift the curve? Simulate and plot improvement curves with different values of $\alpha$. 

In [ ]:
######################
### YOUR CODE HERE ###
######################